<a href="https://colab.research.google.com/github/carneiro-santos/dataton_fase_5_/blob/main/fase_5_FIAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import pandas as pd

# carregar arquivo
file = "Tabela Geral Sheets-4.xlsx"

df_2022 = pd.read_excel(file, sheet_name="2022")
df_2023 = pd.read_excel(file, sheet_name="2023")
df_2024 = pd.read_excel(file, sheet_name="2024")

In [28]:
colunas_padrao = [
    'INDE','IAN','IDA','IEG','IAA','IPS','IPP','IPV',
    'PEDRA','FASE','ANO'
]

In [29]:
def padronizar(df, ano):
    df = df.copy()

    df['ANO'] = ano

    for col in colunas_padrao:
        if col not in df.columns:
            df[col] = None

    return df

In [30]:
def padronizar(df, ano):
    df = df.copy()

    df['ANO'] = ano

    for col in colunas_padrao:
        if col not in df.columns:
            df[col] = None

    return df[colunas_padrao]

In [31]:
df_2022 = padronizar(df_2022, 2022)
df_2023 = padronizar(df_2023, 2023)
df_2024 = padronizar(df_2024, 2024)

In [32]:
def imputar_por_grupo(df, coluna):
    return df.groupby(['ANO','FASE'])[coluna]\
             .transform(lambda x: x.fillna(x.mean()))

In [33]:
colunas_imputar = ['IAN','IDA','IEG','IAA','IPS']

for col in colunas_imputar:
    df_2022[col] = imputar_por_grupo(df_2022, col)
    df_2023[col] = imputar_por_grupo(df_2023, col)
    df_2024[col] = imputar_por_grupo(df_2024, col)

In [34]:
def tratar_ipp_ipv(df):
    # fase 8 não tem esses indicadores
    df.loc[df['FASE'] == 8, ['IPP','IPV']] = None

    # imputar só para fases válidas
    for col in ['IPP','IPV']:
        df[col] = df.groupby(['ANO','FASE'])[col]\
                    .transform(lambda x: x.fillna(x.mean()))

    return df

In [35]:
df_2022 = tratar_ipp_ipv(df_2022)
df_2023 = tratar_ipp_ipv(df_2023)
df_2024 = tratar_ipp_ipv(df_2024)

In [36]:
def calcular_inde(df):

    # fase 0 a 7
    mask_1 = df['FASE'] <= 7

    df.loc[mask_1, 'INDE'] = (
        df['IAN']*0.1 +
        df['IDA']*0.2 +
        df['IEG']*0.2 +
        df['IAA']*0.1 +
        df['IPS']*0.1 +
        df['IPP']*0.1 +
        df['IPV']*0.2
    )

    # fase 8
    mask_2 = df['FASE'] == 8

    df.loc[mask_2, 'INDE'] = (
        df['IAN']*0.1 +
        df['IDA']*0.4 +
        df['IEG']*0.2 +
        df['IAA']*0.1 +
        df['IPS']*0.2
    )

    return df

In [37]:
def classificar_pedra(inde):
    if inde < 5.506:
        return 'Quartzo'
    elif inde < 6.868:
        return 'Ágata'
    elif inde < 8.230:
        return 'Ametista'
    else:
        return 'Topázio'

In [39]:
import pandas as pd

# carregar arquivo
file = "Tabela Geral Sheets-4.xlsx"

df_2022 = pd.read_excel(file, sheet_name="2022")
df_2023 = pd.read_excel(file, sheet_name="2023")
df_2024 = pd.read_excel(file, sheet_name="2024")

print("Initial df_2022 head:")
display(df_2022.head())
print("\nInitial df_2022 info:")
display(df_2022.info())

colunas_padrao = [
    'INDE','IAN','IDA','IEG','IAA','IPS','IPP','IPV',
    'PEDRA','FASE','ANO'
]

def padronizar(df, ano):
    df = df.copy()

    df['ANO'] = ano

    # Rename existing 'Fase' column to 'FASE' if it exists
    if 'Fase' in df.columns:
        df = df.rename(columns={'Fase': 'FASE'})

    year_suffix = str(ano)[-2:] # e.g., '22', '23', '24'

    # Rename 'INDE' column: check for plain 'INDE' first, then year-suffixed variations
    found_inde_col = None
    if 'INDE' in df.columns:
        found_inde_col = 'INDE'
    else:
        for col in df.columns:
            if (col.startswith('INDE ') and year_suffix in col) or \
               (col.startswith('INDE ') and str(ano) in col):
                found_inde_col = col
                break
    if found_inde_col and found_inde_col != 'INDE':
        df = df.rename(columns={found_inde_col: 'INDE'})

    # Rename 'PEDRA' column: check for plain 'PEDRA' first, then year-suffixed variations
    found_pedra_col = None
    if 'PEDRA' in df.columns:
        found_pedra_col = 'PEDRA'
    else:
        for col in df.columns:
            if (col.startswith('Pedra ') and year_suffix in col) or \
               (col.startswith('Pedra ') and str(ano) in col):
                found_pedra_col = col
                break
    if found_pedra_col and found_pedra_col != 'PEDRA':
        df = df.rename(columns={found_pedra_col: 'PEDRA'})

    for col in colunas_padrao:
        if col not in df.columns:
            df[col] = None

    return df

df_2022 = padronizar(df_2022, 2022)
df_2023 = padronizar(df_2023, 2023)
df_2024 = padronizar(df_2024, 2024)

def imputar_por_grupo(df, coluna):
    return df.groupby(['ANO','FASE'])[coluna]\
             .transform(lambda x: x.fillna(x.mean()))

colunas_imputar = ['IAN','IDA','IEG','IAA','IPS']

for col in colunas_imputar:
    df_2022[col] = imputar_por_grupo(df_2022, col)
    df_2023[col] = imputar_por_grupo(df_2023, col)
    df_2024[col] = imputar_por_grupo(df_2024, col)

def tratar_ipp_ipv(df):
    # fase 8 não tem esses indicadores
    df.loc[df['FASE'] == 8, ['IPP','IPV']] = None

    # imputar só para fases válidas
    for col in ['IPP','IPV']:
        df[col] = df.groupby(['ANO','FASE'])[col]\
                    .transform(lambda x: x.fillna(x.mean()))

    return df

df_2022 = tratar_ipp_ipv(df_2022)
df_2023 = tratar_ipp_ipv(df_2023)
df_2024 = tratar_ipp_ipv(df_2024)

def calcular_inde(df):

    # fase 0 a 7
    mask_1 = df['FASE'] <= 7

    df.loc[mask_1, 'INDE'] = (
        df['IAN']*0.1 +
        df['IDA']*0.2 +
        df['IEG']*0.2 +
        df['IAA']*0.1 +
        df['IPS']*0.1 +
        df['IPP']*0.1 +
        df['IPV']*0.2
    )

    # fase 8
    mask_2 = df['FASE'] == 8

    df.loc[mask_2, 'INDE'] = (
        df['IAN']*0.1 +
        df['IDA']*0.4 +
        df['IEG']*0.2 +
        df['IAA']*0.1 +
        df['IPS']*0.2
    )

    return df

def classificar_pedra(inde):
    if inde < 5.506:
        return 'Quartzo'
    elif inde < 6.868:
        return 'Ágata'
    elif inde < 8.230:
        return 'Ametista'
    else:
        return 'Topázio'

df_2022 = calcular_inde(df_2022)
df_2023 = calcular_inde(df_2023)
df_2024 = calcular_inde(df_2024)

for df_temp in [df_2022, df_2023, df_2024]:
    # Impute any remaining None/NaN values in 'INDE' with the column's mean
    df_temp['INDE'] = df_temp['INDE'].fillna(df_temp['INDE'].mean())
    df_temp['PEDRA'] = df_temp['INDE'].apply(classificar_pedra)

Initial df_2022 head:


,RA,Fase,Turma,Nome,Ano nasc,Idade 22,Gênero,Ano ingresso,Instituição de ensino,Pedra 20,...,Inglês,Indicado,Atingiu PV,IPV,IAN,Fase ideal,Defas,Destaque IEG,Destaque IDA,Destaque IPV
0,RA-1,7,A,Aluno-1,2003,19,Menina,2016,Escola Pública,Ametista,...,6.0,Sim,Não,7.278,5.0,Fase 8 (Universitários),-1,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
1,RA-2,7,A,Aluno-2,2005,17,Menina,2017,Rede Decisão,Ametista,...,9.7,Não,Não,6.778,10.0,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
2,RA-3,7,A,Aluno-3,2005,17,Menina,2016,Rede Decisão,Ametista,...,6.9,Não,Não,7.556,10.0,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Destaque: A sua boa integração aos Princípios ...
3,RA-4,7,A,Aluno-4,2005,17,Menino,2017,Rede Decisão,Ametista,...,8.7,Não,Não,5.278,10.0,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
4,RA-5,7,A,Aluno-5,2005,17,Menina,2016,Rede Decisão,Ametista,...,5.7,Não,Não,7.389,10.0,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...



Initial df_2022 info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 860 entries, 0 to 859
Data columns (total 42 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   RA                     860 non-null    object 
 1   Fase                   860 non-null    int64  
 2   Turma                  860 non-null    object 
 3   Nome                   860 non-null    object 
 4   Ano nasc               860 non-null    int64  
 5   Idade 22               860 non-null    int64  
 6   Gênero                 860 non-null    object 
 7   Ano ingresso           860 non-null    int64  
 8   Instituição de ensino  860 non-null    object 
 9   Pedra 20               323 non-null    object 
 10  Pedra 21               462 non-null    object 
 11  Pedra 22               860 non-null    object 
 12  INDE 22                860 non-null    float64
 13  Cg                     860 non-null    int64  
 14  Cf                     860 non-null

None

/tmp/ipykernel_6511/3362399344.py:85: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .transform(lambda x: x.fillna(x.mean()))


TypeError: '<=' not supported between instances of 'str' and 'int'

In [ ]:
# This cell is now redundant as its logic has been merged into a previous cell.

In [40]:
df_final = pd.concat([df_2022, df_2023, df_2024], ignore_index=True)

In [44]:
df_final.isnull().mean().sort_values(ascending=False)

,0
Destaque IPV.1,1.000000
Avaliador6,0.998020
Avaliador5,0.951155
Inglês,0.906601
Rec Av4,0.902310
...,...
IPS,0.000000
Ano ingresso,0.000000
Gênero,0.000000
ANO,0.000000


In [42]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)

cols = ['IAN','IDA','IEG','IAA','IPS']

df_final[cols] = imputer.fit_transform(df_final[cols])

In [43]:
df_final['DELTA_IDA'] = df_final.groupby('RA')['IDA'].diff()